In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold 
from sklearn.metrics import (roc_auc_score, average_precision_score, 
                             f1_score, accuracy_score, precision_score, 
                             recall_score, confusion_matrix, 
                             precision_recall_curve)
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore, BayesianEstimator
from pgmpy.inference import VariableElimination

# Load and prepare data
data = pd.read_excel(# enter file path)
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Discretization function
def discretize_binary_global(X_df):
    X_discretized = X_df.copy()
    for column in X_discretized.columns:
        median = X_discretized[column].median()
        X_discretized[column] = (X_discretized[column] > median).astype(int)
    return X_discretized

# Apply discretization
X_discretized = discretize_binary_global(X)
selected_feature_indexes = [24, 30, 28, 9, 3, 12, 13, 8, 2, 10, 7, 6, 27, 25, 36, 20, 33, 19, 15]
selected_feature_names = X_discretized.columns[selected_feature_indexes].tolist()
data_selected = X_discretized[selected_feature_names].copy()
data_selected['RRI'] = Y

# Initialize 10-fold CV
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Storage variables
all_actuals = []
all_probs = []
auc_scores = []
auprc_scores = []
fold_actuals = []
fold_probs = []

# Cross-validation loop
for train_index, test_index in kf.split(data_selected.drop('RRI', axis=1), data_selected['RRI']):
    # Rest of the loop remains unchanged
    train_data = data_selected.iloc[train_index]
    test_data = data_selected.iloc[test_index]
    
    # Structure learning with BIC
    scoring_method = BicScore(train_data)
    hc = HillClimbSearch(train_data)
    best_model = hc.estimate(scoring_method=scoring_method)
    
    # Model fitting
    model = BayesianNetwork(best_model.edges())
    model.fit(train_data, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=10)
    
    # Inference setup
    infer = VariableElimination(model)
    y_prob = []
    for _, row in test_data.iterrows():
        evidence = {col: row[col] for col in selected_feature_names 
                    if col != 'RRI' and col in model.nodes()}
        try:
            prob = infer.query(variables=['RRI'], evidence=evidence).values[1]
        except:
            prob = 0.5  # Fallback probability if inference fails
        y_prob.append(prob)
    
    # Store metrics
    y_test = test_data['RRI'].values
    auc = roc_auc_score(y_test, y_prob)
    auprc = average_precision_score(y_test, y_prob)
    
    auc_scores.append(auc)
    auprc_scores.append(auprc)
    all_actuals.extend(y_test.tolist())
    all_probs.extend(y_prob)
    fold_actuals.append(y_test)
    fold_probs.append(y_prob)

# Calculate optimal threshold
precision, recall, thresholds = precision_recall_curve(all_actuals, all_probs)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

# Calculate threshold-based metrics
f1_list, acc_list, prec_list, sens_list, spec_list = [], [], [], [], []

for y_true, y_prob in zip(fold_actuals, fold_probs):
    y_pred = (np.array(y_prob) >= best_threshold).astype(int)
    
    f1_list.append(f1_score(y_true, y_pred))
    acc_list.append(accuracy_score(y_true, y_pred))
    prec_list.append(precision_score(y_true, y_pred, zero_division=0))
    sens_list.append(recall_score(y_true, y_pred))
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    spec_list.append(spec)

# Format results
def format_results(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

print("\nComprehensive Model Performance:")
print("--------------------------------")
print(f"AUC:           {format_results(np.mean(auc_scores), np.std(auc_scores))}")
print(f"AUPRC:         {format_results(np.mean(auprc_scores), np.std(auprc_scores))}")
print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"F1 Score:      {format_results(np.mean(f1_list), np.std(f1_list))}")
print(f"Accuracy:      {format_results(np.mean(acc_list), np.std(acc_list))}")
print(f"Precision:     {format_results(np.mean(prec_list), np.std(prec_list))}")
print(f"Sensitivity:   {format_results(np.mean(sens_list), np.std(sens_list))}")
print(f"Specificity:   {format_results(np.mean(spec_list), np.std(spec_list))}")

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]

...

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]


Comprehensive Model Performance:
--------------------------------
AUC:           0.6411 ± 0.0398
AUPRC:         0.1359 ± 0.0196
Optimal Threshold: 0.1015
F1 Score:      0.2401 ± 0.0401
Accuracy:      0.6837 ± 0.0662
Precision:     0.1555 ± 0.0281
Sensitivity:   0.5423 ± 0.1117
Specificity:   0.6979 ± 0.0789
